In [1]:
import pandas as pd
import numpy as np

In [92]:
df = pd.read_csv("linear_regression_dataset.csv")
df.head()

,AGE,FEMALE,LOS,RACE,TOTCHG,APRDRG
0,17,1,2,1.0,2660,560
1,17,0,2,1.0,1689,753
2,17,1,7,1.0,20060,930
3,17,1,1,1.0,736,758
4,17,1,1,1.0,1194,754


In [131]:
y=df['TOTCHG'].values
x_df=df.drop(columns=['TOTCHG'])

X = df[['AGE','LOS']].values

In [133]:
df.isnull().sum()
df.dropna(inplace=True)
df.isnull().sum()


AGE       0
FEMALE    0
LOS       0
RACE      0
TOTCHG    0
APRDRG    0
dtype: int64

In [206]:
def train_test_split(X, y, test_ratio=0.2):
    
    m = len(y)
    split_idx = int(m * (1 - test_ratio))
    np.random.seed(42)
    indices = np.random.permutation(m)
    
    train_idx = indices[:split_idx]
    test_idx = indices[split_idx:]
    
    return X[train_idx], y[train_idx], X[test_idx], y[test_idx]

In [226]:
X_train_std, train_stats = standardize_data(X_train)
X_test_std, _ = standardize_data(X_test, stats=train_stats)

C:\Users\hardi\AppData\Local\Temp\ipykernel_24624\3944493979.py:10: RuntimeWarning: invalid value encountered in divide
  X_standardized = (X - mu) / sigma
C:\Users\hardi\AppData\Local\Temp\ipykernel_24624\3944493979.py:10: RuntimeWarning: divide by zero encountered in divide
  X_standardized = (X - mu) / sigma


In [271]:
import pandas as pd
import numpy as np

FILE_NAME = 'linear_regression_dataset.csv'

def get_data(filename):
    df = pd.read_csv(filename)
    if 'TOTCHG' not in df.columns:
        raise ValueError("Column TOTCHG not found")
        
    y = df['TOTCHG'].values
    x_raw = df.drop(columns=['TOTCHG'])
    
    cat_cols = [c for c in ['RACE', 'APRDRG', 'FEMALE'] if c in x_raw.columns]
    x_encoded = pd.get_dummies(x_raw, columns=cat_cols, drop_first=True)
    
    return x_encoded.values.astype(float), y

def clean_data(x):
    out = x.copy()
    for i in range(out.shape[1]):
        col = out[:, i]
        if np.isnan(col).any():
            out[np.isnan(col), i] = np.nanmean(col)
    return out

def split_data(x, y, ratio=0.2):
    m = len(y)
    size = int(m * (1 - ratio))
    np.random.seed(47)
    idx = np.random.permutation(m)
    
    return x[idx[:size]], y[idx[:size]], x[idx[size:]], y[idx[size:]]

def scale(x, stats=None):
    if stats is None:
        u = np.mean(x, axis=0)
        s = np.std(x, axis=0)
        if np.ndim(s) == 0:
            if s == 0: s = 1
        else:
            s[s == 0] = 1
        stats = (u, s)
    else:
        u, s = stats
        
    return (x - u) / s, stats

def get_score(y_true, y_pred):
    res = np.sum((y_true - y_pred) ** 2)
    tot = np.sum((y_true - np.mean(y_true)) ** 2)
    if tot == 0: return 0.0
    return 1 - (res / tot)

def fit(x, y, lr, epochs):
    m, n = x.shape
    w = np.zeros(n)
    b = 0.0
    
    print(f"{'Epoch':<10} | {'Cost':<15} | {'R2 Score':<10}")
    print("-" * 45)
    
    for i in range(epochs):
        h = np.dot(x, w) + b
        diff = h - y
        
        grad_w = (1/m) * np.dot(x.T, diff)
        grad_b = (1/m) * np.sum(diff)
        
        w = w - lr * grad_w
        b = b - lr * grad_b
        
        if i % 100 == 0:
            cost = (1 / (2 * m)) * np.sum(diff ** 2)
            r2 = get_score(y, h)
            print(f"{i:<10} | {cost:<15.4f} | {r2:<10.4f}")
            
    return w, b

try:
    x_all, y_all = get_data(FILE_NAME)
    x_all = clean_data(x_all)
    
    x_train, y_train, x_test, y_test = split_data(x_all, y_all)
    
    x_train_s, x_stats = scale(x_train)
    x_test_s, _ = scale(x_test, x_stats)
    
    y_mean = np.mean(y_train)
    y_std = np.std(y_train)
    if y_std == 0: y_std = 1
    y_train_s = (y_train - y_mean) / y_std
    
    alpha = 0.01
    epochs = 1000
    
    w, b = fit(x_train_s, y_train_s, alpha, epochs)
    
    pred_train_s = np.dot(x_train_s, w) + b
    pred_test_s = np.dot(x_test_s, w) + b
    
    pred_train_real = (pred_train_s * y_std) + y_mean
    pred_test_real = (pred_test_s * y_std) + y_mean
    
    print("-" * 45)
    print(f"Final Train R2: {get_score(y_train, pred_train_real):.4f}")
    print(f"Final Test R2:  {get_score(y_test, pred_test_real):.4f}")

except Exception as e:
    print(e)

Epoch      | Cost            | R2 Score  
---------------------------------------------
0          | 0.5000          | 0.0000    
100        | 0.0667          | 0.8666    
200        | 0.0275          | 0.9449    
300        | 0.0214          | 0.9571    
400        | 0.0197          | 0.9606    
500        | 0.0188          | 0.9624    
600        | 0.0182          | 0.9636    
700        | 0.0178          | 0.9644    
800        | 0.0175          | 0.9651    
900        | 0.0172          | 0.9655    
---------------------------------------------
Final Train R2: 0.9659
Final Test R2:  0.8323


[Errno 2] No such file or directory: 'hospital_costs.'
